#### We will perform all the necessary steps. But we do not need to create new DB or Vector stores, because we just want to add more things, right

What you can do, you can create your chunks and create the embeddings out of it, and then you need to say I want to add this chunks.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API key is set.")

API key is set.


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

#### **STEP 1: Extracting Text from PDFs**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./Docs/Doodle_Final_Report.pdf")

docs = loader.load()

docs


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-13T01:25:39+00:00', 'author': '', 'keywords': '', 'moddate': '2025-05-13T01:25:39+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': './Docs/Doodle_Final_Report.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Doodle Recognition Using Deep Learning\nDarpankumar Jiyani\nDeep Learning\nDepartment of Applied Data Science\nSID: 017536623\ndarpankumarpareshbhai.jiyani@sjsu.edu\nVansh Sharma\nDeep Learning\nDepartment of Applied Data Science\nSID: 016003624\nvansh.sharma@sjsu.edu\nAbstract—We set out to build a sketch-friendly AI that does\ntwo jobs at once: recognize whatever a user is doodling and, in the\nsame breath, offer a quick tip on how to polish the drawing. We\ntrained two vision models on a balanced slice of Google’s Quick\nDraw! 

### **Creating our own Metadata for PDF chunks**

In [6]:
for i in docs:    #List have capability to make in place changes. So we can add the metadata to each document in the list.

    i.metadata = {"source": "DL Final Report.pdf",
                  "People": "Me and Darshan"}

#### **STEP 2: Splitting the Document into CHUNKS**

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'DL Final Report.pdf', 'People': 'Me and Darshan'}, page_content='Doodle Recognition Using Deep Learning\nDarpankumar Jiyani\nDeep Learning\nDepartment of Applied Data Science\nSID: 017536623\ndarpankumarpareshbhai.jiyani@sjsu.edu\nVansh Sharma\nDeep Learning\nDepartment of Applied Data Science\nSID: 016003624\nvansh.sharma@sjsu.edu\nAbstract—We set out to build a sketch-friendly AI that does\ntwo jobs at once: recognize whatever a user is doodling and, in the\nsame breath, offer a quick tip on how to polish the drawing. We\ntrained two vision models on a balanced slice of Google’s Quick\nDraw! (approximately 345000 images across 345 classes). The\nlightweight EfficientNetV2-B0 proved the more reliable partner,\nhitting about 75% top-1 accuracy on sketches it had never seen\nwhile keeping response times under 100ms. A larger Vision\nTransformer scored higher in-sample but cracked when faced\nwith fresh scribbles classic overfitting so we sidelined it. To t

In [8]:
len(chunks)

32

In [9]:
chunks[0].metadata

{'source': 'DL Final Report.pdf', 'People': 'Me and Darshan'}

#### **STEP 3: Creating Embeddings for the Chunks**

In [10]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

#### **STEP 4: Create and Store Embeddings in the EXISTING Local Vector Store**

Now here we need to make some changes from the File 3.0

In [14]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="./DB/", embedding_function=embedding_model)

In [15]:
vectorstore.add_documents(chunks)

['82a0c225-c390-4452-9b22-a12424a2c360',
 'fe24dd63-b0ce-4216-a1a1-7df75dd26748',
 '9b5ceb53-ccad-492c-8826-24f3efa0405f',
 '27824fae-065b-42bb-af8a-81a21edfe326',
 '5e5911ef-818b-4c14-84ee-c72452a47a99',
 'ad12243d-93ac-4c36-af96-8d43d608cf9b',
 '00639ac1-8669-42a9-b914-657a0edc6200',
 '4888037d-4813-4873-be9c-a9ed33d2bc9c',
 'e235c5e8-0566-4de1-b326-8275fa517bb7',
 '8f769ebf-15fb-4a2d-9985-ff93b1084637',
 '657063b6-4560-4ea7-8535-d0de8a31743a',
 'dc244cb5-d222-4fad-be99-b5cdf286e129',
 '815d2fe1-4e2c-4a64-8e8d-236c0d913df1',
 '550264c9-cacb-4b9f-8026-6b0681f2180f',
 'f8adbd7e-ef56-4869-a9de-65bd239b70e2',
 'b713aad8-6739-428c-83fa-2c91369ebc51',
 '30dddc19-dd4c-4d36-8bee-f73cd76000a8',
 '277176ae-9147-4f2c-8d0b-1f7b78544dbe',
 '800c6ef2-6ed6-4786-adb3-95dac21a7b44',
 '59321dab-a724-4745-99ca-48f8b28e31fe',
 '7eba8672-e4d1-4177-a486-879b657854e4',
 '85e41dfb-827b-4edb-8bd7-51ba491440da',
 'cd877929-91dd-4c78-9063-523b2235fa9a',
 'a137f620-d1b2-44d8-9c66-42bbfc672421',
 '7ac1446c-6ea6-

#### **STEP 5: Semantic Search**

In [17]:
vectorstore.similarity_search("What is work being done in Doodle project?", k=3)

[Document(metadata={'People': 'Me and Darshan', 'source': 'DL Final Report.pdf'}, page_content='Doodle Recognition Using Deep Learning\nDarpankumar Jiyani\nDeep Learning\nDepartment of Applied Data Science\nSID: 017536623\ndarpankumarpareshbhai.jiyani@sjsu.edu\nVansh Sharma\nDeep Learning\nDepartment of Applied Data Science\nSID: 016003624\nvansh.sharma@sjsu.edu\nAbstract—We set out to build a sketch-friendly AI that does\ntwo jobs at once: recognize whatever a user is doodling and, in the\nsame breath, offer a quick tip on how to polish the drawing. We\ntrained two vision models on a balanced slice of Google’s Quick\nDraw! (approximately 345000 images across 345 classes). The\nlightweight EfficientNetV2-B0 proved the more reliable partner,\nhitting about 75% top-1 accuracy on sketches it had never seen\nwhile keeping response times under 100ms. A larger Vision\nTransformer scored higher in-sample but cracked when faced\nwith fresh scribbles classic overfitting so we sidelined it. To t

See above, now we are getting chunks, and you can verify the metadata that we defined earlier. So we got our data from new pdf.

So this how We I can persist the RAG information locally. I can use that RAG database again and again. I can add more documents. I can change the meta data, I can do so much of things. I have full control

## **This is RAG**